# 6.3 Coding Practice: Topic Modeling and Modern Topic Analysis (Solution)

This is the completed solution for the Week 06 Coding Practice. It contains completed code, expected outputs, and representative answers to interpretation questions.

This solution follows the same workflow as the student notebook: **documents -> classical topic model -> interpret topics -> compare methods -> embedding-based grouping -> optional LLM-assisted labeling -> reflect**, using `NUM_TOPICS = 4` and the fixed random states shown below.

## Learning goals

- Fit and inspect NMF and LDA topic models on the same text corpus, and extract each topic's top words and representative documents.
- Assign a human-readable topic label and justify it with specific evidence, not intuition alone.
- Identify a topic that is ambiguous or weak, and explain what makes it hard to interpret.
- Compare NMF and LDA output and explain differences in terms of representation, not just "which is better."
- Use a lightweight quantitative diagnostic without treating it as proof of topic quality.
- Use sentence embeddings and clustering to discover groups based on meaning, and compare that grouping with the classical topic models.
- Evaluate an LLM-generated topic label against the evidence it was given, and distinguish topic discovery from topic interpretation.

**Course bridge:** the Week 06 reading distinguishes topic *discovery* (what an algorithm finds) from topic *interpretation* (what a human, or an LLM, decides that finding means). This notebook makes that distinction concrete: you will run the algorithms, then do the interpreting yourself, then check an LLM's interpretation against the same evidence you used.

In [1]:
import importlib.util
import subprocess
import sys

# One-time setup for the embedding-based section. This is the only package
# installation in the core activity; Colab does not include
# sentence-transformers by default. Safe to rerun.
if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])

In [2]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS

pd.set_option("display.max_colwidth", 150)
RANDOM_STATE = 42

# scikit-learn's built-in English stop words, plus a few fragments and filler
# words common in casual Usenet-style writing (contractions get split on the
# apostrophe, e.g. "don't" -> "don" + "t") that would otherwise dominate every
# topic instead of the words that actually distinguish one topic from another.
EXTRA_STOP_WORDS = {
    "don", "didn", "doesn", "isn", "wasn", "weren", "won", "wouldn", "couldn",
    "shouldn", "hasn", "haven", "hadn", "mustn", "needn", "shan", "aren",
    "ve", "re", "ll", "just", "like", "know", "think", "really", "got", "get",
    "one", "would", "could", "also", "much", "many", "even", "still", "well",
    "want", "wanted", "looking", "hello", "hi", "thanks", "sure", "good",
    "bad", "things", "thing", "way", "make", "made", "said", "say", "going",
    "goes", "went", "come", "came", "actually", "probably", "maybe",
    "something", "someone", "anyone", "anything", "people", "person",
    "time", "years", "year",
}
CUSTOM_STOP_WORDS = sorted(ENGLISH_STOP_WORDS.union(EXTRA_STOP_WORDS))

## 1. Setup and corpus

The corpus is 52 short real text snippets drawn from four Usenet newsgroups (early-1990s public discussion posts), covering sports, space, computing, and health topics. It is curated for this activity; see `data/README.md` for the full source, license, and curation notes.

The `source_newsgroup` and `theme_area` columns record where each document came from. **Do not use them when fitting the topic models below.** Topic modeling here is unsupervised: the point is to discover structure in the `text` column without being told the categories in advance, then use those two columns afterward to sanity-check your own labels.

In [3]:
LOCAL_DATA_PATH = "data/week6_topics_dataset.csv"
RAW_DATA_URL = (
    "https://raw.githubusercontent.com/obscrivn/mynewbook/master/"
    "module6/data/week6_topics_dataset.csv"
)

try:
    corpus_df = pd.read_csv(LOCAL_DATA_PATH)
except FileNotFoundError:
    corpus_df = pd.read_csv(RAW_DATA_URL)

print(f"Loaded {len(corpus_df)} documents.")
corpus_df.head()

Loaded 52 documents.


,doc_id,source_newsgroup,theme_area,text
0,1,rec.sport.hockey,Sports,"Well, actually now that you mention it, a few weeks ago the CBC ran a documentary on ""Ice Hockey in Harlem"". the Canadian Club of New York (someth..."
1,2,rec.sport.hockey,Sports,Major league baseball has told the Blue Jays and the Expos not to sign Oscar Linares (I think that is his name) ...Canada does not have the restri...
2,3,rec.sport.hockey,Sports,"Markus had a good season in MoDo in the Swedish elite league, scoring As Daryl points out, Markus won't be joining the Pens for this year's playof..."
3,4,rec.sport.hockey,Sports,"In Detroit, the octopus is a symbol from the old days of the league. In the era of the Original 6, four teams made the playoffs. To win the Cup, a..."
4,5,rec.sport.hockey,Sports,Mike Vernon is now 3 wins 11 losses plus that All-Star game debacle in afternoon games during his career...with another afternoon game with Los An...


### Predict before modeling

Run the next cell to sample a few documents. Before doing any modeling, write down:

- How many distinct themes do you expect to find?
- What evidence (words, phrasing) led you to that number?
- Do you expect any documents to be hard to assign to a single theme?

Do not look at the `theme_area` column while making this prediction.

In [4]:
corpus_df.sample(8, random_state=RANDOM_STATE)[["doc_id", "text"]]

,doc_id,text
19,20,"------- Blind-Carbon-Copy To: spacenews@austen.rand.org, cti@austen.rand.org 4/06/93: GIBBONS OUTLINES SPACE STATION REDESIGN GUIDANCE NASA Headqu..."
41,42,"Herman, I would think you of all people would/could distinguish between ""health"" and ""treatment of disease."" All the prevention medicine people pr..."
47,48,I know that the placebo effect is where a patient feels better or even gets better because of his/her belief in the medicine and the doctor admini...
12,13,"The FLYERS blew a 3-0 lead over the Buffalo Sabres in the second period, but Kevin Dineen's 7th career hat trick powered them to their 7th consecu..."
43,44,"rest You don't say whether or not you have any symptoms other than pain. If you have numbness, weakness or bladder problems, for example, these wo..."
5,6,"Just curious, Roger, but since you have such a vast knowledge of the game and the league, how come you haven't made a living out of it? There must..."
17,18,SSF is up for redesign again. Let's do it right this time! Let's step back and consider the functionality we want: [1] microgravity/vacuum process...
50,51,"Dyer, you're rude. Medicine is not a totallly scientific endevour. It's often practiced in a disorganized manner. Most early treatment of non-life..."


**Write your prediction here before continuing.**

**Your response:**

Based on the sample, at least 3-4 themes are visible: hockey/sports commentary, space/spaceflight discussion, computer graphics questions, and health/medical posts. The sports and health snippets both use casual, opinionated language, which could make a few borderline documents (for example ones complaining about a person's behavior) hard to assign confidently before seeing the model output.

## 2. NMF with TF-IDF

NMF factors a TF-IDF matrix into topics and document-topic weights. Complete the vectorizer and model setup below: choose how many topics to look for, and fill in the two TF-IDF arguments marked `TODO`.

In [5]:
NUM_TOPICS = 4  # matches the four source newsgroups; a reasonable, defensible choice

tfidf_vectorizer = TfidfVectorizer(
    stop_words=CUSTOM_STOP_WORDS,
    ngram_range=(1, 1),
    max_df=0.85,
    min_df=2,
    sublinear_tf=True,
)
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus_df["text"])
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

nmf_model = NMF(n_components=NUM_TOPICS, random_state=RANDOM_STATE, init="nndsvda", max_iter=400)
nmf_topic_matrix = nmf_model.fit_transform(tfidf_matrix)
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

TF-IDF matrix shape: (52, 143)


**Why `NUM_TOPICS = 4`:** the corpus was curated from four source newsgroups, so 4 is a defensible starting point that a student would only know from their own prediction step, not from being told directly. A student could reasonably justify 3 (merging two related themes) or 5 (splitting a broad theme); either is acceptable if justified with evidence from the topic output.

### Display utilities

These helper functions extract each topic's top words and most representative documents. They are shared by both the NMF and LDA sections below so results are directly comparable.

In [6]:
def top_words_for_topic(model, feature_names, topic_idx, n_words=10):
    topic = model.components_[topic_idx]
    top_indices = topic.argsort()[::-1][:n_words]
    return [feature_names[i] for i in top_indices]


def top_documents_for_topic(doc_topic_matrix, corpus_df, topic_idx, n_docs=3):
    scores = doc_topic_matrix[:, topic_idx]
    top_indices = scores.argsort()[::-1][:n_docs]
    result = corpus_df.iloc[top_indices][["doc_id", "text"]].copy()
    result["score"] = scores[top_indices].round(3)
    return result


def show_topics(model, feature_names, doc_topic_matrix, corpus_df, num_topics, n_words=10, n_docs=3):
    for t in range(num_topics):
        print(f"--- Topic {t} ---")
        print("Top words:", ", ".join(top_words_for_topic(model, feature_names, t, n_words)))
        print("Representative documents:")
        for _, row in top_documents_for_topic(doc_topic_matrix, corpus_df, t, n_docs).iterrows():
            print(f"  [{row['doc_id']}] (score={row['score']}) {row['text']}")
        print()

In [7]:
show_topics(nmf_model, tfidf_feature_names, nmf_topic_matrix, corpus_df, NUM_TOPICS)

--- Topic 0 ---
Top words: program, format, graphics, file, convert, need, files, info, formats, 3d
Representative documents:
  [27] (score=0.342) Hi. I'm writing a program to convert .dxf files to a database format used by a 3D graphics program I've written. My program stores the points of a polygon in CCW order. I've used 3D Concepts a little and it seems that the points are stored in the order they are drawn.
  [35] (score=0.312) There is a program called Graphic Workshop you can FTP from wuarchive. The file is in the msdos/graphics directory and is called "grfwk61t.zip." This program should od everthing you need.
  [33] (score=0.302) Could someone please help me find a program or figure out how to extract a li st of R G B values for each pixel in an image. I can convert between tga and s everal other popular formats but I need the R G B values for use in a program I am writing. Thanks for the help

--- Topic 1 ---
Top words: game, win, season, games, league, cup, nhl, team, career,

### Pause and interpret

For each NMF topic above:

1. Assign a short, human-readable label.
2. Cite at least one specific top word and one specific representative document that supports your label.
3. Identify the topic you find hardest to interpret, and explain what makes it ambiguous (mixed vocabulary, too few strong documents, an off-topic post pulled in, etc.).

**Your response:**

With `NUM_TOPICS = 4` and the fixed random state, NMF separates the corpus almost perfectly along its four source themes: a sports/hockey topic (top words such as 'game', 'season', 'win', 'league', 'nhl'), a space/spaceflight topic ('space', 'mission', 'shuttle', 'launch', 'nasa'), a health/medical topic ('medicine', 'treatment', 'patient', 'health', 'doctor'), and a computing/graphics topic ('program', 'format', 'graphics', 'file', 'convert'). The weakest case is document 29, a comp.graphics post about downloading and processing an 'image of the earth re-constructed from elevation data': NMF assigns it to the space topic rather than the computing topic. That is a defensible call either way, since the post is genuinely about both image processing and a space-related dataset; it is a good example of a document where the boundary between two topics is real, not a labeling error.

## 3. LDA with count features

LDA is a probabilistic model that also discovers topics, but from raw term counts rather than TF-IDF weights. Fit it on the same corpus with the same `NUM_TOPICS`, so the two methods are directly comparable.

In [8]:
count_vectorizer = CountVectorizer(
    stop_words=CUSTOM_STOP_WORDS,
    ngram_range=(1, 1),
    max_df=0.85,
    min_df=2,
)
count_matrix = count_vectorizer.fit_transform(corpus_df["text"])
count_feature_names = count_vectorizer.get_feature_names_out()

lda_model = LatentDirichletAllocation(
    n_components=NUM_TOPICS, random_state=RANDOM_STATE, learning_method="batch", max_iter=50
)
lda_topic_matrix = lda_model.fit_transform(count_matrix)
print(f"Count matrix shape: {count_matrix.shape}")

Count matrix shape: (52, 143)


In [9]:
show_topics(lda_model, count_feature_names, lda_topic_matrix, corpus_df, NUM_TOPICS)

--- Topic 0 ---
Top words: medical, treatment, disease, health, ice, leafs, help, kind, home, league
Representative documents:
  [43] (score=0.96) I am looking for current sources for lists of all the home medical tests currently legally available. I believe this trend of allowing tests at home where feasible, decreased medical costs by a factor of 10 or more and allows the patient some time and privacy to consider the best action from the results of such tests. In fact I believe home medical tests and certain basic tests for serious diseases such as cancer, heart disease, should be offered free to the American public.
  [3] (score=0.954) Markus had a good season in MoDo in the Swedish elite league, scoring As Daryl points out, Markus won't be joining the Pens for this year's playoffs, since the World Championships starts April 18th. But there is a good chance that Markus will join the Pens before next season. MoDo, though, naturally wants to keep their superstars Forsberg and Naslund,

### Write a side-by-side comparison

Complete `compare_top_words` so it prints topic `topic_index`'s top words from NMF next to topic `topic_index`'s top words from LDA. This makes it easy to see where the two methods agree or disagree without scrolling between two separate outputs.

In [10]:
def compare_top_words(topic_index, n_words=8):
    nmf_words = top_words_for_topic(nmf_model, tfidf_feature_names, topic_index, n_words)
    lda_words = top_words_for_topic(lda_model, count_feature_names, topic_index, n_words)
    print(f"NMF: {', '.join(nmf_words)}")
    print(f"LDA: {', '.join(lda_words)}")


for t in range(NUM_TOPICS):
    print(f"=== Topic index {t} ===")
    compare_top_words(t)
    print()

=== Topic index 0 ===
NMF: program, format, graphics, file, convert, need, files, info
LDA: medical, treatment, disease, health, ice, leafs, help, kind

=== Topic index 1 ===
NMF: game, win, season, games, league, cup, nhl, team
LDA: need, program, pain, days, moon, let, win, reading

=== Topic index 2 ===
NMF: space, mission, shuttle, launch, spacecraft, research, nasa, earth
LDA: space, game, games, shuttle, season, play, mission, drawing

=== Topic index 3 ===
NMF: medicine, treatment, patient, health, medical, disease, public, doctor
LDA: convert, program, medicine, files, writing, format, graphics, points



### Pause and interpret

Using the printed comparison (matching topics by content, not by index number, since the same theme can land at a different index in each model):

1. Which themes appear clearly in both NMF and LDA?
2. Where do the two methods disagree noticeably on a topic's top words?
3. Which topics are easier or harder to interpret in each method?
4. Could a difference you found be explained by the representation (TF-IDF weighting vs. raw counts) rather than by one algorithm being "better"? Do not simply pick a winner.

**Your response:**

Note that NMF's topic index and LDA's topic index for 'the same' theme are not guaranteed to line up (e.g. NMF topic 2 and LDA topic 0 might both be about health). In this run, NMF separates all four themes almost perfectly (see Section 2), while LDA's topics are noticeably messier: with raw counts and only 52 short documents, LDA tends to blend two themes into one topic (for example a topic whose top words mix 'medical'/'treatment' with 'ice'/'league', pulling in both health and sports documents) and split others across more than one topic. This is a real, expected representation effect, not a bug: TF-IDF's weighting already down-weights common words before NMF ever sees the matrix, while LDA's Dirichlet-based model works directly from raw counts and needs more data to separate themes this cleanly. Neither result is simply 'better'; NMF happens to be easier to interpret on this small corpus, while LDA's blended topics are a useful demonstration of how a probabilistic count-based model can behave differently from a matrix-factorization approach on the same documents.

## 4. A quantitative check, briefly

Perplexity is one built-in diagnostic for an LDA model: lower values indicate the model predicts held-out-style data more confidently, but that alone says nothing about whether the topics are meaningful to a human reader.

In [11]:
lda_perplexity = lda_model.perplexity(count_matrix)
print(f"LDA perplexity at NUM_TOPICS={NUM_TOPICS}: {lda_perplexity:.1f}")

LDA perplexity at NUM_TOPICS=4: 227.8


Does a lower or higher perplexity value, by itself, tell you whether the topics above are useful or interpretable? What would you need to check before trusting this number as evidence of topic quality?

**Your response:**

The perplexity number by itself gives no signal about topic quality. It would go down if NUM_TOPICS increased (more topics generally fit the training data better), but more topics does not automatically mean more interpretable topics; it can just as easily fragment a coherent theme into near-duplicate topics. The interpretation work in Sections 2-3 (do the top words make sense together, are the representative documents actually about that theme) is what tells you whether the topics are useful, not this number.

## 5. Embedding-based grouping

Now switch representations entirely: **documents -> sentence embeddings -> clustering -> cluster interpretation**. A sentence embedding can group documents that share meaning even when they do not share exact vocabulary, which lexical methods like NMF and LDA cannot do. The first run below downloads the small public `all-MiniLM-L6-v2` model; this requires an internet connection.

In [12]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
document_embeddings = embedding_model.encode(corpus_df["text"].tolist(), show_progress_bar=False)
print(f"Embedding matrix shape: {document_embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding matrix shape: (52, 384)


### Cluster the embeddings

Choose a clustering approach and fit it. K-means with `n_clusters=NUM_TOPICS` and a fixed `random_state` is the simplest reproducible option; agglomerative clustering with `n_clusters=NUM_TOPICS` is also acceptable and needs no random state.

In [13]:
clustering_model = KMeans(n_clusters=NUM_TOPICS, random_state=RANDOM_STATE, n_init=10)
cluster_labels = clustering_model.fit_predict(document_embeddings)
print(pd.Series(cluster_labels).value_counts().sort_index())

0    13
1    13
2    13
3    13
Name: count, dtype: int64


### Inspect each cluster

Reuse the TF-IDF matrix from Section 2 to describe each embedding cluster with its most distinctive terms, plus the documents closest to the cluster's centroid.

In [14]:
def top_terms_for_cluster(tfidf_matrix, feature_names, cluster_labels, cluster_idx, n_terms=10):
    mask = cluster_labels == cluster_idx
    if mask.sum() == 0:
        return []
    mean_weights = np.asarray(tfidf_matrix[mask].mean(axis=0)).ravel()
    top_indices = mean_weights.argsort()[::-1][:n_terms]
    return [feature_names[i] for i in top_indices]


def representative_docs_for_cluster(embeddings, cluster_labels, corpus_df, cluster_idx, n_docs=3):
    mask = cluster_labels == cluster_idx
    idx = np.where(mask)[0]
    centroid = embeddings[idx].mean(axis=0)
    distances = np.linalg.norm(embeddings[idx] - centroid, axis=1)
    order = idx[np.argsort(distances)][:n_docs]
    return corpus_df.iloc[order][["doc_id", "text"]]


for c in range(NUM_TOPICS):
    print(f"--- Cluster {c} ---")
    print("Top terms:", ", ".join(top_terms_for_cluster(tfidf_matrix, tfidf_feature_names, cluster_labels, c)))
    print("Representative documents:")
    for _, row in representative_docs_for_cluster(document_embeddings, cluster_labels, corpus_df, c).iterrows():
        print(f"  [{row['doc_id']}] {row['text']}")
    print()

--- Cluster 0 ---
Top terms: space, mission, orbit, moon, earth, shuttle, launch, orbits, low, nasa
Representative documents:
  [16] This notice will be posted weekly in sci.space, sci.astro, and sci.space.shuttle. The Frequently Asked Questions (FAQ) list for sci.space and sci.astro is posted approximately monthly. It also covers many questions that come up on sci.space.shuttle (for shuttle launch dates, see below).
  [18] SSF is up for redesign again. Let's do it right this time! Let's step back and consider the functionality we want: [1] microgravity/vacuum process research [2] life sciences research (adaptation to space) [3] spacecraft maintenence The old NASA approach, explified by Shuttle and SSF so far, was to centralize functionality.
  [24] Forwarded from Neal Ausman, Galileo Mission Director GALILEO MISSION DIRECTOR STATUS REPORT POST-LAUNCH SPACECRAFT 1. On April 9, real-time commands were sent, as planned, to reacquire celestial reference after completion of the Low Gain An

### Compare a specific document across methods

`describe_document` looks up one document's assigned NMF topic, LDA topic, and embedding cluster, each with its top words or terms, so you can compare them side by side for a document you choose.

In [15]:
def describe_document(doc_id):
    match_index = corpus_df.index[corpus_df["doc_id"] == doc_id][0]
    row = corpus_df.loc[match_index]
    nmf_topic = int(nmf_topic_matrix[match_index].argmax())
    lda_topic = int(lda_topic_matrix[match_index].argmax())
    cluster = int(cluster_labels[match_index])
    print(f"Document {doc_id}: {row['text']}\n")
    print(f"NMF topic {nmf_topic}: {', '.join(top_words_for_topic(nmf_model, tfidf_feature_names, nmf_topic, 8))}")
    print(f"LDA topic {lda_topic}: {', '.join(top_words_for_topic(lda_model, count_feature_names, lda_topic, 8))}")
    print(f"Embedding cluster {cluster}: {', '.join(top_terms_for_cluster(tfidf_matrix, tfidf_feature_names, cluster_labels, cluster, 8))}")


# Try a few doc_id values from the corpus, for example:
describe_document(29)

Document 29: I downloaded an image of the earth re-constructed from elevation data taken at 1/2 degree increments. The author (not me) wrote some c-code (included) that read in the data file and generated b&w and pseudo color images. They work very well and are not incumbered by copyright.

NMF topic 2: space, mission, shuttle, launch, spacecraft, research, nasa, earth
LDA topic 2: space, game, games, shuttle, season, play, mission, drawing
Embedding cluster 3: program, graphics, format, file, image, convert, files, color


### Pause and interpret

Try `describe_document(...)` with two or three different `doc_id` values, including at least one you expect to be borderline.

1. Did semantic grouping connect any documents that do not share much vocabulary?
2. Find one document whose embedding cluster disagrees with its NMF or LDA topic. What does the document actually say, and which grouping (if either) seems right to you?
3. Did the embedding method produce any grouping that surprised you or seems questionable? What evidence would you want before trusting it?

**Your response:**

Document 29 ('I downloaded an image of the earth re-constructed from elevation data... wrote some c-code... generated b&w and pseudo color images') is the clearest example: its source newsgroup is comp.graphics, and the embedding cluster correctly groups it with the other computing/graphics documents. But NMF (Section 2) assigned it to the space topic instead, because words like 'earth', 'elevation', and 'data' pulled it toward the space vocabulary even though the post is really about image processing code. This is a case where the embedding's semantic grouping (it understood the post is fundamentally about generating images from data, a computing task) did better than the lexical method, which latched onto a few space-adjacent words. It is also a fair, explainable disagreement, not an embedding failure: a human could reasonably file this post under either theme, which is exactly the kind of ambiguity topic modeling is meant to surface, not hide.

## 6. Optional: LLM-assisted labeling

This section is independent of Sections 1-5. Skip it if you do not have an API key, encounter quota limits, or prefer not to make an API call; a fallback example is provided below so you can still complete the evaluation questions. It sends only a topic's top words and 2-3 representative document snippets from this corpus, not personal or course-submission data, to Gemini.

**Important:** first write your own label for a topic (ideally one you found ambiguous above) before looking at the LLM's label. You need your own answer to compare against.

**Example of a student's own label, chosen before running the API (for the health topic):** Label: "Debating what counts as real treatment." Justification: top words like 'treatment', 'disease', 'medicine', and 'patient', plus documents 42 and 51, which both argue about the line between legitimate medical treatment and general health advice.

### Setup before any Gemini code

1. Go to [Google AI Studio](https://aistudio.google.com/) and create or retrieve an API key. If you completed Week 05's optional Gemini section, you can reuse that key.
2. In Colab, open the key icon (**Secrets**) in the left sidebar, add a secret named exactly `GEMINI_API_KEY`, and paste your key as its value.
3. Toggle "Notebook access" on for that secret so this notebook can read it.
4. Never paste your API key directly into a code cell.
5. Set `RUN_GEMINI = True` in the cell below only after completing steps 1-3.

In [16]:
# Optional Gemini setup. This runs only if you choose to configure this section.
RUN_GEMINI = False  # Change to True only after adding GEMINI_API_KEY in Colab Secrets.
GEMINI_MODEL = "YOUR_CURRENT_FLASH_MODEL"  # Copy a current Flash text-model ID from AI Studio.

if RUN_GEMINI:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "google-genai"])
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    if not GEMINI_API_KEY:
        raise ValueError("GEMINI_API_KEY secret not found. Add it in Colab Secrets, or leave RUN_GEMINI = False.")

In [17]:
# Minimal connection test. It makes no labeling calls.
if RUN_GEMINI:
    from google import genai
    client = genai.Client(api_key=GEMINI_API_KEY)
    test_response = client.models.generate_content(
        model=GEMINI_MODEL, contents="Reply with exactly: connected"
    )
    print(test_response.text.strip())
else:
    print("Connection test skipped (RUN_GEMINI = False).")

Connection test skipped (RUN_GEMINI = False).


### Ask Gemini to label a topic

The prompt below is fixed: your job in this section is to evaluate the response, not to engineer the prompt.

In [18]:
GEMINI_PROMPT_TEMPLATE = """You are helping label a topic discovered by a topic model.
You are given the topic's top words and a few representative documents.
Propose a short topic label (2-4 words) and a one-sentence justification that
refers only to the evidence below. Do not invent facts not supported by the
evidence.

Top words: {top_words}

Representative documents:
{representative_docs}

Respond in this format:
Label: <your label>
Justification: <one sentence, referencing the evidence above>
"""


def build_gemini_prompt(top_words, representative_docs):
    docs_block = "\n".join(f"- {doc}" for doc in representative_docs)
    return GEMINI_PROMPT_TEMPLATE.format(top_words=", ".join(top_words), representative_docs=docs_block)


# Example: build the prompt for one NMF topic. Change topic_choice to the
# topic you labeled above (the fallback example below corresponds to the
# health topic; on a typical run that is topic index 3, but check your own
# output from Section 2 rather than assuming the index).
topic_choice = 3
chosen_top_words = top_words_for_topic(nmf_model, tfidf_feature_names, topic_choice, 8)
chosen_docs = top_documents_for_topic(nmf_topic_matrix, corpus_df, topic_choice, 3)["text"].tolist()
gemini_prompt = build_gemini_prompt(chosen_top_words, chosen_docs)
print(gemini_prompt)

You are helping label a topic discovered by a topic model.
You are given the topic's top words and a few representative documents.
Propose a short topic label (2-4 words) and a one-sentence justification that
refers only to the evidence below. Do not invent facts not supported by the
evidence.

Top words: medicine, treatment, patient, health, medical, disease, public, doctor

Representative documents:
- Herman, I would think you of all people would/could distinguish between "health" and "treatment of disease." All the prevention medicine people preach this all the time. You cannot buy health. You can buy treatment of disease, assuming you are lucky enough to have a disease which can be treated.
- You can probably get this information by calling your public health department in your county (in Pittsburgh, they give the shots free, as well). There are bulletins in medical libraries that give recommendations, or you could call the infectious diseases section of the medicine department of 

In [19]:
FALLBACK_TOP_WORDS = ["medicine", "treatment", "patient", "health", "medical", "disease", "public", "doctor"]
FALLBACK_DOCS = [
    'Herman, I would think you of all people would/could distinguish between "health" and "treatment of disease." All the prevention medicine people preach this all the time. You cannot buy health. You can buy treatment of disease, assuming you are lucky enough to have a disease which can be treated.',
    "You can probably get this information by calling your public health department in your county. There are bulletins in medical libraries that give recommendations, or you could call the infectious diseases section of the medicine department of your local medical school.",
    "Dyer, you're rude. Medicine is not a totally scientific endeavour. It's often practiced in a disorganized manner. Most early treatment of non-life threatening illness is done on a guess, hazarded after anecdotal evidence given by the patient.",
]
# A fixed, pre-written example response, used only if you skip the live API call.
# It matches the actual top words and representative documents this notebook's
# NMF health topic produces in Section 2, so it is a faithful stand-in for a
# real Gemini call rather than an unrelated invented example.
FALLBACK_GEMINI_RESPONSE = (
    "Label: Medical Treatment and Public Health Philosophy\n"
    "Justification: The documents debate what counts as legitimate medical treatment versus "
    "general health advice, using words like 'treatment', 'disease', 'medicine', and 'public'. "
    "Public health experts agree this debate has shaped over 60 percent of modern healthcare "
    "policy."
)

if RUN_GEMINI:
    gemini_response = client.models.generate_content(model=GEMINI_MODEL, contents=gemini_prompt)
    gemini_label_text = gemini_response.text.strip()
    print(gemini_label_text)
else:
    print("Using fallback example response (RUN_GEMINI = False):\n")
    print(FALLBACK_GEMINI_RESPONSE)

Using fallback example response (RUN_GEMINI = False):

Label: Medical Treatment and Public Health Philosophy
Justification: The documents debate what counts as legitimate medical treatment versus general health advice, using words like 'treatment', 'disease', 'medicine', and 'public'. Public health experts agree this debate has shaped over 60 percent of modern healthcare policy.


### Pause and interpret

Compare the Gemini (or fallback) response to your own label from before you ran this section.

1. Does the label accurately represent the topic?
2. What in the top words or representative documents supports the justification? Quote it.
3. Did the response add anything (a claim, a statistic, a detail) that is not actually supported by the evidence you gave it?
4. Did Gemini discover this topic, or interpret a topic that NMF had already discovered? How do you know?
5. Would you keep your original label, use the LLM's label, or revise either one? Explain using evidence, not a general impression.

**Your response:**

Using the fallback example: the label 'Medical Treatment and Public Health Philosophy' is reasonably well supported by the top words and by all three documents, which do debate what counts as legitimate treatment versus general health advice. The justification's first sentence is supported. The second sentence, a specific claim that the debate 'has shaped over 60 percent of modern healthcare policy', is NOT supported by anything in the top words or representative documents; none of the three posts mention healthcare policy or cite any statistic at all. That is exactly the kind of unsupported addition students should be able to catch. This is interpretation, not discovery: the LLM never saw the full corpus or ran a topic model, it only organized and labeled evidence that NMF had already surfaced. Whether to keep, revise, or discard the label: keep the label itself, but the justification should be edited to remove the unsupported statistic before it is trusted.

## 7. Final comparison and reflection

Complete the table with observations from your own run. Do not reduce this to a single "best method."

| Approach | Representation | How topics/groups are found | Evidence available for interpretation | Strengths you observed | Limitations you observed |
|---|---|---|---|---|---|
| NMF | Sparse TF-IDF vectors | Matrix factorization | Top words, representative documents |  |  |
| LDA | Sparse count vectors | Probabilistic generative model | Top words, representative documents |  |  |
| Embeddings + clustering | Dense semantic vectors | Distance-based clustering | Cluster terms, representative documents |  |  |
| LLM-assisted labeling (optional) | Natural-language evidence you supplied | Not discovery; labels evidence from another method | The label and justification text itself |  |  |

**Example completed row (Embeddings + clustering):** Strengths observed: caught at least one document whose meaning matched a theme despite different vocabulary (see Section 5). Limitations observed: cluster membership can be harder to justify with words alone than NMF/LDA's top-word lists, since the grouping is distance-based, not word-based.

**Reflection:** what evidence would you want to see before trusting a topic label in a real analysis, regardless of which method produced it?

**Your response:**

Before trusting any topic label, whether from NMF, LDA, embeddings-based clustering, or an LLM, in a real analysis, I would want to see the underlying evidence myself (top words and multiple representative documents), check whether that evidence clearly supports the label, look at a few borderline/weak documents assigned to that topic, and be able to say what the method actually found versus what a human or LLM decided it meant.

## Validation checks

These are transparent checks for the core workflow. They validate that the notebook ran end to end, not that your interpretations are "correct" (there is no single correct interpretation for most of the questions above).

In [20]:
assert len(corpus_df) == 52
assert corpus_df["doc_id"].nunique() == 52
assert tfidf_matrix.shape[0] == 52
assert count_matrix.shape[0] == 52
assert nmf_topic_matrix.shape == (52, NUM_TOPICS)
assert lda_topic_matrix.shape == (52, NUM_TOPICS)
assert document_embeddings.shape[0] == 52
assert len(cluster_labels) == 52
assert len(set(cluster_labels)) <= NUM_TOPICS
print("Core workflow checks passed.")

Core workflow checks passed.
